# CLEIDS-Edge — Notebook 04: Baseline Models

Trains and evaluates 8 baseline models (Random Forest, SVM, Standalone CNN, Standalone LSTM, and 4 re-implemented published lightweight/hybrid IDS papers) on all 5 benchmark datasets (NSL-KDD, CICIDS2017, UNSW-NB15, TON_IoT, IoT-23), binary task, for the head-to-head comparison in thesis Chapter 4. Reuses the exact same `data/processed/<dataset>/{train,val,test}.npz` splits CLEIDS-Edge itself trained on — no re-splitting, no re-sampling.

**GPU runtime (L4) required** for the 6 deep-learning baselines (Standalone CNN/LSTM + the 4 published re-implementations); Random Forest and SVM run on CPU but there is no need to switch runtimes mid-notebook.

**Run this notebook directly in Colab's browser UI**, not through a proxied kernel connection — same reasoning as Notebook 03 (`google.colab.userdata.get()` requires the real Colab frontend for its permission handshake).

**Evaluation protocol — stated as a fixed rule, applied identically to every model/dataset combination, no case-by-case judgment calls**:
1. Threshold calibration (validation-set max-F1 search over 99 candidate cutoffs) is applied to **every** binary evaluation, unconditionally.
2. If a model's **default-threshold (0.5) FPR exceeds 0.20** on a given dataset, that specific model/dataset combination is **automatically** retrained with extended patience (`patience=10` instead of 5) before final numbers are reported. This is a programmatic check, not a judgment call — see `§7` below. (0.20 is the same cutoff formalized for CLEIDS-Edge's own Notebook 03 results, chosen because it cleanly separates that project's actual good-FPR runs, ≤0.08, from its bad-FPR runs, ≥0.42 — see `CLEIDS_PROJECT_BRIEF.md`.)
3. Both default-threshold and tuned-threshold metrics are reported for every combination, matching how CLEIDS-Edge's own results are reported (`main_results.json` + `tuned_threshold_results.json`).

**Re-implementation fidelity (stated per baseline, no fabricated hyperparameters)**: Standalone CNN/LSTM are exact ablations of CLEIDS-Edge's own architecture. Altaie & Hoomod (2024) is fully verified from its open-access paper. Nazir (2024) and Misrak & Melaku (2025) are paywalled — only abstract-level detail was verifiable, so their architectures use documented reasonable defaults. Wang (2023) is open access but its own paper doesn't publish fixed hyperparameters (Optuna-tuned per dataset instead) — also a documented reasonable default. See `src/models.py` docstrings and each baseline's markdown cell below for exactly what is verified vs. assumed.

**No fabricated numbers for any baseline, ever** — every metric below comes from a real `model.fit()`/`model.evaluate()` on this project's actual data, not approximated from the original papers' own reported numbers (which used different datasets/splits and aren't directly comparable).

**Incremental backup**: given a real mid-training disconnect happened during Notebook 03, every individual run's results/checkpoint is backed up to Drive immediately, and a GitHub push happens after each baseline model type finishes across all 5 datasets — so a disconnect costs at most one baseline's retraining, not the whole notebook.

## 1. Repo setup (clone/pull + auth)

In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/NehlTech/CLEIDS-Edge.git"
REPO_DIR = "/content/CLEIDS-Edge"

GITHUB_TOKEN = None
try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
except Exception as e:
    print(f"[DEBUG] userdata.get('GITHUB_TOKEN') raised {type(e).__name__}: {e}")
    GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN")

if not GITHUB_TOKEN:
    raise RuntimeError(
        "GITHUB_TOKEN not found. Add it as a Colab secret (key icon in the left sidebar) "
        "if running in the real Colab UI, or set os.environ['GITHUB_TOKEN'] manually for "
        "this session if running over a proxied connection."
    )

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

AUTH_REMOTE = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@")
subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", AUTH_REMOTE], check=True)

subprocess.run(["git", "-C", REPO_DIR, "config", "user.email", "obololastkiller@gmail.com"])
subprocess.run(["git", "-C", REPO_DIR, "config", "user.name", "Bright Adu-Boahene"])

os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())


## 2. Google Drive mount (source for processed data; backup target for model checkpoints/results/figures)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/CLEIDS_Edge"
DRIVE_MODELS = os.path.join(DRIVE_ROOT, "models")
DRIVE_RESULTS = os.path.join(DRIVE_ROOT, "results")
DRIVE_FIGURES = os.path.join(DRIVE_ROOT, "figures")
DRIVE_DATA_PROCESSED = os.path.join(DRIVE_ROOT, "data_processed")
for d in (DRIVE_MODELS, DRIVE_RESULTS, DRIVE_FIGURES):
    os.makedirs(d, exist_ok=True)
print("Drive ready at:", DRIVE_ROOT)


## 3. Data bridge — copy processed splits from Google Drive

In [ ]:
import shutil

DATASETS = ["nsl-kdd", "cicids2017", "unsw-nb15", "ton-iot", "iot-23"]

if not os.path.isdir(DRIVE_DATA_PROCESSED):
    raise RuntimeError(
        f"{DRIVE_DATA_PROCESSED} not found. Upload data/processed/ to Google Drive at "
        f"MyDrive/CLEIDS_Edge/data_processed/ first (same data Notebook 03 used)."
    )

for name in DATASETS:
    src_dir = os.path.join(DRIVE_DATA_PROCESSED, name)
    dst_dir = f"data/processed/{name}"
    os.makedirs(dst_dir, exist_ok=True)
    for fn in ["train.npz", "val.npz", "test.npz", "label_classes.json", "feature_names.json"]:
        src = os.path.join(src_dir, fn)
        dst = os.path.join(dst_dir, fn)
        if not os.path.exists(src):
            raise RuntimeError(f"{src} not found on Drive. Re-upload data/processed/{name}/ and re-run.")
        if os.path.exists(dst):
            print(f"[skip] {dst} already present")
            continue
        shutil.copy2(src, dst)
        print(f"[copied] {dst} ({os.path.getsize(dst)/1e6:.1f} MB)")

manifest_dst = "data/processed/preprocessing_manifest.json"
if not os.path.exists(manifest_dst):
    shutil.copy2(os.path.join(DRIVE_DATA_PROCESSED, "preprocessing_manifest.json"), manifest_dst)

print("\nAll processed data copied from Drive.")


## 4. Setup & GPU Verification

In [ ]:
import sys
import time
import json
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix,
)

tf.random.set_seed(42)
np.random.seed(42)

gpus = tf.config.list_physical_devices("GPU")
if not gpus:
    raise RuntimeError(
        "CRITICAL ERROR: No GPU runtime detected! Notebook 04 requires a GPU (L4) for the "
        "6 deep-learning baselines. Switch runtime to GPU (Runtime -> Change runtime type)."
    )
print(f"[HARDWARE OK] GPU visible: {gpus}")
print("TensorFlow version:", tf.__version__)

sys.path.insert(0, os.path.join(REPO_DIR, "src"))
from models import (
    build_random_forest, build_svm, build_standalone_cnn, build_standalone_lstm,
    build_nazir2024_hybrid, build_altaie_hoomod2024, build_wang2023_dlbilstm, build_misrak_melaku2025,
)

for d in ["models", "results", "figures"]:
    os.makedirs(d, exist_ok=True)

with open("data/processed/preprocessing_manifest.json") as f:
    prep_manifest = json.load(f)

with open("results/main_results.json") as f:
    cleids_edge_results = json.load(f)
print("Loaded CLEIDS-Edge's own results (reference only, not modified):", list(cleids_edge_results.keys()))


## 5. Evaluation Utilities

Shared across all 8 baselines — one generic `train_and_evaluate_baseline()` handles both sklearn models (RF, SVM) and Keras models (the 6 deep-learning baselines), so the evaluation protocol (threshold tuning, the FPR>0.20 retry rule, metrics, model size/latency) is applied identically regardless of model type.

In [ ]:
def validate_against_manifest(dataset_name, train_data, val_data, test_data):
    expected = prep_manifest["datasets"][dataset_name]["shapes"]
    actual = {"train": train_data["X_cnn"].shape[0], "val": val_data["X_cnn"].shape[0], "test": test_data["X_cnn"].shape[0]}
    for split, actual_n in actual.items():
        expected_n = expected[split][0]
        if actual_n != expected_n:
            raise RuntimeError(
                f"[{dataset_name}] {split} row count mismatch: manifest says {expected_n:,}, "
                f"loaded {actual_n:,}. Stopping rather than proceeding on mismatched data."
            )


def calculate_fpr(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    return float(fp / (fp + tn)) if (fp + tn) > 0 else 0.0


def compute_binary_metrics(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "threshold": float(threshold),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "auc": float(roc_auc_score(y_true, y_prob)),
        "fpr": calculate_fpr(y_true, y_pred),
    }


def tune_threshold_on_validation(y_val, val_prob):
    """Max-F1 search over 99 candidate thresholds on the VALIDATION set only --
    never touches test-set labels when selecting the threshold, same held-out
    discipline as Notebook 03."""
    thresholds = np.linspace(0.01, 0.99, 99)
    best_t, best_f1 = 0.5, -1.0
    for t in thresholds:
        f1 = f1_score(y_val, (val_prob >= t).astype(int), zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return float(best_t)


def get_model_size_and_latency(model, model_type, X_sample):
    """Rough sanity-check only -- NOT the official CPU-only rigor of Notebook 06's
    latency benchmark, just enough to confirm CLEIDS-Edge's later edge-latency
    comparison has something real to compare against."""
    if model_type == "sklearn":
        import pickle
        n_params = None
        size_mb = len(pickle.dumps(model)) / 1e6
    else:
        n_params = model.count_params()
        size_mb = n_params * 4 / 1e6  # float32 weights, rough estimate

    n_latency_samples = min(100, X_sample.shape[0])
    sample = X_sample[:n_latency_samples]
    t0 = time.time()
    if model_type == "sklearn":
        model.predict_proba(sample)
    else:
        model.predict(sample, batch_size=1, verbose=0)
    elapsed = time.time() - t0
    latency_ms_per_sample = (elapsed / n_latency_samples) * 1000
    return {"n_params": n_params, "size_mb": round(size_mb, 3), "latency_ms_per_sample": round(latency_ms_per_sample, 4)}


def backup_run_to_drive(model_name, dataset_name, ckpt_path):
    """Incremental per-run Drive backup -- cheap (a few seconds), durable
    (Drive survives a Colab disconnect unlike /content). Run after every
    single baseline/dataset combination, not just once at the end."""
    if ckpt_path and os.path.exists(ckpt_path):
        shutil.copy2(ckpt_path, os.path.join(DRIVE_MODELS, os.path.basename(ckpt_path)))
    results_path = "results/baseline_results.json"
    if os.path.exists(results_path):
        shutil.copy2(results_path, os.path.join(DRIVE_RESULTS, "baseline_results.json"))


def push_checkpoint_to_github(commit_message):
    """Periodic GitHub push -- called after each baseline model type finishes
    across all 5 datasets (not after every single run, to avoid commit-spam),
    per the incremental-backup plan. Safe against gitignore silently blocking
    small .keras files, same fix already applied for CLEIDS-Edge's own checkpoints."""
    subprocess.run(["git", "-C", REPO_DIR, "add", "-A", "models/", "results/", "figures/"], check=False)
    commit_res = subprocess.run(
        ["git", "-C", REPO_DIR, "commit", "-m", commit_message], capture_output=True, text=True,
    )
    print(commit_res.stdout, commit_res.stderr)
    if commit_res.returncode == 0:
        subprocess.run(["git", "-C", REPO_DIR, "push", "origin", "HEAD"], check=True)
        print(f"Pushed: {commit_message}")
    else:
        print("Nothing new to commit (or commit failed) -- see output above.")


def train_and_evaluate_baseline(model_name, dataset_name, model_type, build_fn=None, sklearn_fn=None,
                                  epochs=50, batch_size=256, patience=5, fpr_retry_threshold=0.20):
    """Generic train+evaluate for both sklearn (model_type='sklearn') and Keras
    (model_type='keras') baselines. Applies the evaluation protocol identically:
    threshold tuning always, FPR>0.20 auto-retry (Keras models only -- RF/SVM
    have no 'patience'/epoch concept to extend) for every combination, no
    case-by-case exceptions."""
    print("\n" + "=" * 70)
    print(f"[TRAINING] {model_name} on {dataset_name}")
    print("=" * 70)

    data_dir = os.path.join("data/processed", dataset_name)
    train_data = np.load(os.path.join(data_dir, "train.npz"))
    val_data = np.load(os.path.join(data_dir, "val.npz"))
    test_data = np.load(os.path.join(data_dir, "test.npz"))
    validate_against_manifest(dataset_name, train_data, val_data, test_data)

    y_train = train_data["y_bin"].astype(int)
    y_val = val_data["y_bin"].astype(int)
    y_test = test_data["y_bin"].astype(int)

    ckpt_path = None
    retried = False
    retry_note = None

    if model_type == "sklearn":
        X_train, X_val, X_test = train_data["X_flat"], val_data["X_flat"], test_data["X_flat"]
        t0 = time.time()
        model = sklearn_fn()
        model.fit(X_train, y_train)
        train_time_sec = time.time() - t0
        epochs_run = None
        val_prob = model.predict_proba(X_val)[:, 1]
        test_prob = model.predict_proba(X_test)[:, 1]
        default_metrics = compute_binary_metrics(y_test, test_prob, 0.5)
    else:
        X_train, X_val, X_test = train_data["X_cnn"], val_data["X_cnn"], test_data["X_cnn"]
        input_dim = X_train.shape[1]
        ckpt_path = f"models/{model_name}_{dataset_name}_binary.keras"

        def _fit_once(p):
            m = build_fn(input_dim)
            cbs = [
                tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=p, restore_best_weights=True),
                tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6),
                tf.keras.callbacks.ModelCheckpoint(filepath=ckpt_path, monitor="val_loss", save_best_only=True, verbose=0),
            ]
            t0 = time.time()
            hist = m.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=epochs,
                         batch_size=batch_size, callbacks=cbs, verbose=1)
            elapsed = time.time() - t0
            m = tf.keras.models.load_model(ckpt_path)
            return m, hist, elapsed

        model, history, train_time_sec = _fit_once(patience)
        epochs_run = len(history.history["loss"])
        val_prob = model.predict(X_val, batch_size=512, verbose=0).ravel()
        test_prob = model.predict(X_test, batch_size=512, verbose=0).ravel()
        default_metrics = compute_binary_metrics(y_test, test_prob, 0.5)

        if default_metrics["fpr"] > fpr_retry_threshold:
            retried = True
            fpr_before = default_metrics["fpr"]
            print(f"[AUTO-RETRY] default-threshold FPR={fpr_before:.4f} > {fpr_retry_threshold} -- "
                  f"retraining {model_name}/{dataset_name} with patience=10 (rule applied "
                  f"programmatically, not a case-by-case decision).")
            model, history2, retrain_time_sec = _fit_once(10)
            epochs_run = len(history2.history["loss"])
            train_time_sec += retrain_time_sec
            val_prob = model.predict(X_val, batch_size=512, verbose=0).ravel()
            test_prob = model.predict(X_test, batch_size=512, verbose=0).ravel()
            default_metrics = compute_binary_metrics(y_test, test_prob, 0.5)
            retry_note = {"fpr_before_retry": fpr_before, "fpr_after_retry": default_metrics["fpr"]}
            print(f"[AUTO-RETRY] FPR after retry: {default_metrics['fpr']:.4f} (was {fpr_before:.4f})")

    best_threshold = tune_threshold_on_validation(y_val, val_prob)
    tuned_metrics = compute_binary_metrics(y_test, test_prob, best_threshold)

    size_info = get_model_size_and_latency(model, model_type, X_test)

    result = {
        "default_threshold": default_metrics,
        "tuned_threshold": tuned_metrics,
        "train_time_sec": round(train_time_sec, 2),
        "epochs_run": epochs_run,
        "fpr_retry_triggered": retried,
        "retry_note": retry_note,
        "checkpoint_path": ckpt_path,
        **size_info,
    }
    print(f"[{model_name}/{dataset_name}] default: Acc={default_metrics['accuracy']:.4f} FPR={default_metrics['fpr']:.4f} | "
          f"tuned(t={best_threshold:.2f}): Acc={tuned_metrics['accuracy']:.4f} F1={tuned_metrics['f1']:.4f} FPR={tuned_metrics['fpr']:.4f}")

    backup_run_to_drive(model_name, dataset_name, ckpt_path)
    return result


def save_baseline_results(baseline_results):
    """Additive load-then-update -- never overwrites results already saved
    from a prior partial run of this notebook."""
    path = "results/baseline_results.json"
    existing = {}
    if os.path.exists(path):
        with open(path) as f:
            existing = json.load(f)
    for model_name, per_dataset in baseline_results.items():
        existing.setdefault(model_name, {}).update(per_dataset)
    with open(path, "w") as f:
        json.dump(existing, f, indent=2)
    print(f"Wrote {path} ({len(existing)} models total)")
    return existing


## 6. SVM kernel-vs-linear decision (data-driven, not a silent downgrade)

Kernel SVC scales roughly O(n^2)-O(n^3) with training rows. A 50,000-row threshold is used (consistent with this project's other 50k-scale decisions, e.g. the CICIDS2017/IoT-23 SMOTE cap) — any dataset's post-SMOTE training set above that uses `LinearSVC + CalibratedClassifierCV` instead of kernel `SVC`, documented explicitly per dataset below rather than silently substituted.

In [ ]:
SVM_LINEAR_FALLBACK = {}
print(f"{'Dataset':<12} | {'Train rows':>10} | {'SVM variant':<12}")
print("-" * 40)
for name in DATASETS:
    n_train = prep_manifest["datasets"][name]["shapes"]["train"][0]
    use_linear = n_train > 50_000
    SVM_LINEAR_FALLBACK[name] = use_linear
    variant = "linear (fallback)" if use_linear else "kernel (rbf)"
    print(f"{name:<12} | {n_train:>10,} | {variant:<12}")


## 7. Random Forest

In [ ]:
baseline_results = {"random_forest": {}}
t0 = time.time()
for name in DATASETS:
    baseline_results["random_forest"][name] = train_and_evaluate_baseline(
        "random_forest", name, model_type="sklearn", sklearn_fn=build_random_forest,
    )
rf_elapsed = time.time() - t0
save_baseline_results(baseline_results)
print(f"\nRandom Forest across all 5 datasets: {rf_elapsed/60:.1f} min total.")


## 8. SVM

In [ ]:
baseline_results = {"svm": {}}
t0 = time.time()
for name in DATASETS:
    baseline_results["svm"][name] = train_and_evaluate_baseline(
        "svm", name, model_type="sklearn",
        sklearn_fn=lambda n=name: build_svm(use_linear_fallback=SVM_LINEAR_FALLBACK[n]),
    )
svm_elapsed = time.time() - t0
save_baseline_results(baseline_results)
push_checkpoint_to_github("Notebook 04: Random Forest + SVM baselines (all 5 datasets)")
print(f"\nSVM across all 5 datasets: {svm_elapsed/60:.1f} min total.")


## 9. Time estimate — STOP AND REVIEW before the 6 GPU-based baselines

Random Forest and SVM are CPU-based and fast; the remaining 6 baselines (Standalone CNN, Standalone LSTM, and the 4 published re-implementations) are GPU deep-learning models, each trained across all 5 datasets — 30 runs total, comparable in scale to all of Notebook 03. Review the estimate below before continuing; if the projected total exceeds a few hours, decide now whether to let it run unattended (e.g. overnight) or step in.

In [ ]:
classical_ml_elapsed = rf_elapsed + svm_elapsed
print(f"[ESTIMATE] Random Forest + SVM (10 runs, CPU): {classical_ml_elapsed/60:.1f} min actual.")
print(f"[ESTIMATE] Remaining: 6 GPU-based baselines x 5 datasets = 30 runs.")
print(f"[ESTIMATE] Notebook 03's own 10 GPU runs (CLEIDS-Edge) took ~4-5 hours total wall-clock "
      f"(including retries). This notebook has 3x as many GPU runs (30 vs 10), so a naive "
      f"projection is roughly 3x that: ~12-15 hours -- though architectures here are mostly "
      f"simpler (single-branch ablations, or comparably lightweight published designs) than "
      f"CLEIDS-Edge's own hybrid, so actual time may be lower. Some fraction of these will also "
      f"trigger the FPR>0.20 auto-retry, adding to the total unpredictably, same as Notebook 03.")
print("[ESTIMATE] STOP AND REVIEW: decide now whether to let this run unattended (e.g. overnight) "
      "or proceed interactively, per the run instructions.")


## 10. Standalone CNN

Exact CNN-branch ablation of CLEIDS-Edge's own architecture (identical Conv1D 64->128 filter sizes) with `GlobalAveragePooling1D` + Dense replacing the LSTM branch. Same training protocol (epochs=50, batch_size=256) as CLEIDS-Edge itself, for a fair comparison. Fully faithful by construction — no published paper to diverge from.

In [ ]:
baseline_results = {"standalone_cnn": {}}
t0 = time.time()
for name in DATASETS:
    baseline_results["standalone_cnn"][name] = train_and_evaluate_baseline(
        "standalone_cnn", name, model_type="keras", build_fn=build_standalone_cnn,
        epochs=50, batch_size=256, patience=5,
    )
cnn_elapsed = time.time() - t0
save_baseline_results(baseline_results)
print(f"\nStandalone CNN across all 5 datasets: {cnn_elapsed/60:.1f} min total.")


## 11. Standalone LSTM

Exact LSTM-branch ablation of CLEIDS-Edge's own architecture (identical LSTM(100) size), applied directly to the raw feature vector. Same training protocol as CLEIDS-Edge itself.

In [ ]:
baseline_results = {"standalone_lstm": {}}
t0 = time.time()
for name in DATASETS:
    baseline_results["standalone_lstm"][name] = train_and_evaluate_baseline(
        "standalone_lstm", name, model_type="keras", build_fn=build_standalone_lstm,
        epochs=50, batch_size=256, patience=5,
    )
lstm_elapsed = time.time() - t0
save_baseline_results(baseline_results)
push_checkpoint_to_github("Notebook 04: Standalone CNN + LSTM ablation baselines (all 5 datasets)")
print(f"\nStandalone LSTM across all 5 datasets: {lstm_elapsed/60:.1f} min total.")


## 12. Nazir et al. (2024) Hybrid CNN-LSTM

**Fidelity statement**: Ain Shams Engineering Journal, 15, 102777 is **paywalled** (ScienceDirect) — real full-text access attempts failed (403 Forbidden via direct fetch and mirror pages). Only abstract-level detail was verifiable: GWO (Gray Wolf Optimizer) feature selection, SMOTE class balancing, MinMax/standard normalization, evaluated on ToN_IoT and UNSW-NB15. **The exact layer sizes, dropout rates, optimizer, and epoch count are NOT published in any source I could access** — the architecture in `src/models.py::build_nazir2024_hybrid` (2 Conv1D blocks at 32/64 filters -> LSTM(128) -> Dense(64)) is a reasonable default reflecting a hybrid CNN-LSTM topology, not a verified reproduction. Trained here with this project's standard protocol (epochs=50, batch_size=256) since the paper's own values aren't available.

In [ ]:
baseline_results = {"nazir2024": {}}
t0 = time.time()
for name in DATASETS:
    baseline_results["nazir2024"][name] = train_and_evaluate_baseline(
        "nazir2024", name, model_type="keras", build_fn=build_nazir2024_hybrid,
        epochs=50, batch_size=256, patience=5,
    )
nazir_elapsed = time.time() - t0
save_baseline_results(baseline_results)
print(f"\nNazir et al. (2024) across all 5 datasets: {nazir_elapsed/60:.1f} min total.")


## 13. Altaie & Hoomod (2024) Hybrid Lightweight CNN+LSTM

**Fidelity statement**: Eng. Technol. Appl. Sci. Res., 14, 16740-16743 is **fully open access** — read directly from the source PDF. Architecture follows the paper's Figure 3 exactly: `Conv1D(64, kernel=8) -> LeakyReLU -> BatchNorm -> MaxPool -> LSTM(256) -> Conv1D(64, kernel=8) -> LeakyReLU -> BatchNorm -> MaxPool -> LSTM(64) -> LSTM(32) -> Dense(256) -> Dense(64) -> output`. **Hyperparameters used here are the paper's own published values (Table I)**: epochs=30, batch_size=32, learning_rate=0.001 (baked into the model's own `.compile()` call), dropout=0.3 — not this project's default 50/256, since the paper specifies its own. Not reproduced: the paper's PRESENT+SPECK feature-encryption step between its two phases — a lightweight-cryptography detail orthogonal to classification accuracy, out of scope for a pure model-comparison benchmark.

In [ ]:
baseline_results = {"altaie_hoomod2024": {}}
t0 = time.time()
for name in DATASETS:
    baseline_results["altaie_hoomod2024"][name] = train_and_evaluate_baseline(
        "altaie_hoomod2024", name, model_type="keras", build_fn=build_altaie_hoomod2024,
        epochs=30, batch_size=32, patience=5,
    )
altaie_elapsed = time.time() - t0
save_baseline_results(baseline_results)
push_checkpoint_to_github("Notebook 04: Nazir2024 + AltaieHoomod2024 baselines (all 5 datasets)")
print(f"\nAltaie & Hoomod (2024) across all 5 datasets: {altaie_elapsed/60:.1f} min total.")


## 14. Wang et al. (2023) "DL-BiLSTM"

**Fidelity statement**: PeerJ Computer Science (DOI 10.7717/peerj-cs.1569) is **open access**, but the paper itself doesn't publish fixed hyperparameters — it tunes BiLSTM/DNN unit counts per-dataset via Optuna rather than reporting fixed values. Confirmed real details: "dual hidden layer DNN" + BiLSTM fusion, IPCA feature reduction, post-training 8-bit dynamic quantization. **Architecture used here** (`Bidirectional(LSTM(64)) -> Dense(128) -> Dense(64) -> output`) is a reasonable default consistent with the paper's stated topology, not a literal reproduction of an Optuna-searched configuration that was never fixed in the source. IPCA feature reduction is NOT re-derived here (this project's shared feature set is fixed across all baselines per the fair-comparison protocol); the 8-bit quantization is out of scope for this notebook (quantization is Notebook 05's focus, applied to CLEIDS-Edge specifically).

In [ ]:
baseline_results = {"wang2023_dlbilstm": {}}
t0 = time.time()
for name in DATASETS:
    baseline_results["wang2023_dlbilstm"][name] = train_and_evaluate_baseline(
        "wang2023_dlbilstm", name, model_type="keras", build_fn=build_wang2023_dlbilstm,
        epochs=50, batch_size=256, patience=5,
    )
wang_elapsed = time.time() - t0
save_baseline_results(baseline_results)
print(f"\nWang et al. (2023) DL-BiLSTM across all 5 datasets: {wang_elapsed/60:.1f} min total.")


## 15. Misrak & Melaku (2025) Lightweight IDS

**Fidelity statement**: Discover Internet of Things, 5, 97 is **paywalled** (Springer) — real full-text access attempts failed (redirect to an authentication gate). Only abstract-level detail was verifiable: RAL-MIFS + two-stage IPCA feature selection, QAT/PTDQ quantization, a "DNN-BiLSTMQ" model evaluated on CIC-IDS2017/CIC-IoT2023. The "DNN-BiLSTMQ" name and abstract both indicate this extends Wang (2023)'s DNN-BiLSTM design (baseline 7) with improved feature engineering and quantization-aware training — **no independently-verified architecture difference could be confirmed**, so the same topology as `build_wang2023_dlbilstm` is reused here. QAT/PTDQ quantization is out of scope for this notebook's standard-training comparison (Notebook 05's focus for CLEIDS-Edge specifically) — reasonable default, not verified from source.

In [ ]:
baseline_results = {"misrak_melaku2025": {}}
t0 = time.time()
for name in DATASETS:
    baseline_results["misrak_melaku2025"][name] = train_and_evaluate_baseline(
        "misrak_melaku2025", name, model_type="keras", build_fn=build_misrak_melaku2025,
        epochs=50, batch_size=256, patience=5,
    )
misrak_elapsed = time.time() - t0
save_baseline_results(baseline_results)
push_checkpoint_to_github("Notebook 04: Wang2023 DL-BiLSTM + MisrakMelaku2025 baselines (all 5 datasets)")
print(f"\nMisrak & Melaku (2025) across all 5 datasets: {misrak_elapsed/60:.1f} min total.")


## 16. Consolidated Results & Retry-Rule Transparency Report

In [ ]:
with open("results/baseline_results.json") as f:
    all_baseline_results = json.load(f)

print("Models with saved results:", list(all_baseline_results.keys()))

print("\n" + "=" * 90)
print("FPR>0.20 AUTO-RETRY RULE -- TRANSPARENCY REPORT")
print("(applies only to the 6 Keras baselines -- RF/SVM have no patience/epoch concept to extend)")
print("=" * 90)
any_retried = False
for model_name, per_dataset in all_baseline_results.items():
    for dataset_name, r in per_dataset.items():
        if r.get("fpr_retry_triggered"):
            any_retried = True
            note = r["retry_note"]
            print(f"[{model_name}/{dataset_name}] RETRIED: default-threshold FPR "
                  f"{note['fpr_before_retry']:.4f} -> {note['fpr_after_retry']:.4f} after patience=10 retrain")
if not any_retried:
    print("No model/dataset combination triggered the FPR>0.20 rule.")


## 17. Backup to Drive + Push to GitHub

In [ ]:
shutil.copy2("results/baseline_results.json", os.path.join(DRIVE_RESULTS, "baseline_results.json"))
for fn in os.listdir("figures"):
    shutil.copy2(os.path.join("figures", fn), os.path.join(DRIVE_FIGURES, fn))
for fn in os.listdir("models"):
    shutil.copy2(os.path.join("models", fn), os.path.join(DRIVE_MODELS, fn))
print("Final Drive backup complete at", DRIVE_ROOT)

push_checkpoint_to_github("Notebook 04: final baseline results consolidation (all 8 models, 5 datasets)")


## 18. Final Consolidated Comparison Table (9 models x 5 datasets)

In [ ]:
MODEL_DISPLAY_ORDER = [
    "cleids_edge", "random_forest", "svm", "standalone_cnn", "standalone_lstm",
    "nazir2024", "altaie_hoomod2024", "wang2023_dlbilstm", "misrak_melaku2025",
]

print("\n" + "=" * 100)
print("CLEIDS-Edge vs. Baselines -- Notebook 04 Headline Comparison (tuned-threshold F1, binary task)")
print("=" * 100)
header = f"{'Model':<20} | " + " | ".join(f"{d:<12}" for d in DATASETS)
print(header)
print("-" * len(header))
for model_name in MODEL_DISPLAY_ORDER:
    if model_name == "cleids_edge":
        row_vals = []
        for d in DATASETS:
            f1 = cleids_edge_results.get(d, {}).get("binary", {}).get("f1")
            row_vals.append(f"{f1:.4f}" if f1 is not None else "N/A")
    else:
        per_dataset = all_baseline_results.get(model_name, {})
        row_vals = []
        for d in DATASETS:
            r = per_dataset.get(d)
            row_vals.append(f"{r['tuned_threshold']['f1']:.4f}" if r else "N/A")
    print(f"{model_name:<20} | " + " | ".join(f"{v:<12}" for v in row_vals))
print("=" * 100)
print("Full details (default + tuned metrics, model size, latency, retry notes) are in results/baseline_results.json.")
print("CLEIDS-Edge's own results (unmodified reference) are in results/main_results.json.")
